**Purpose:** `v2_tfidf_embeddings.ipynb` — part of `02_sentiment/news` (see the stage README).

**Inputs:** `/kaggle/input/news-sentiment/dataset.parquet`

**Outputs:** `results/{combo_key}_learning_curve.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.seeds import SEED_SENTIMENT_NEWS_V2_TFIDF_EMBEDDINGS


https://www.kaggle.com/code/hugoverssimo/v2-tfidfemb/notebook?scriptVersionId=295100980

In [ ]:
!pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118

!pip install transformers==4.36.2 scikit-learn pandas numpy

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected")

# =========================================================
# 5. Sector Descriptions
# =========================================================

# https://www.msci.com/downloads/documents/indexes/gics/GICS+Sector+Definitions+2023.pdf

sector_desc = {
    "Energy": (
        "The Energy Sector comprises companies engaged in exploration & "
        "production, refining & marketing, and storage & transportation of oil "
        "and gas and coal & consumable fuels. It also includes companies that "
        "offer oil & gas equipment and services."
    ),
    "Materials": (
        "The Materials Sector includes companies that manufacture chemicals, "
        "construction materials, forest products, glass, paper and related "
        "packaging products, and metals, minerals and mining companies, "
        "including producers of steel."
    ),
    "Industrials": (
        "The Industrials Sector includes manufacturers and distributors of "
        "capital goods such as aerospace & defense, building products, "
        "electrical equipment and machinery and companies that offer "
        "construction & engineering services. It also includes providers of "
        "commercial & professional services and transportation services."
    ),
    "Consumer Discretionary": (
        "The Consumer Discretionary Sector encompasses those businesses that "
        "tend to be the most sensitive to economic cycles. Its manufacturing "
        "segment includes automobiles & components, household durable goods, "
        "leisure products, and textiles & apparel. The services segment "
        "includes hotels, restaurants, and other leisure facilities, as well as "
        "distributors and retailers of consumer discretionary products."
    ),
    "Consumer Staples": (
        "The Consumer Staples Sector comprises companies whose businesses are "
        "less sensitive to economic cycles. It includes manufacturers and "
        "distributors of food, beverages and tobacco and producers of non-durable "
        "household goods and personal products. It also includes distributors "
        "and retailers of consumer staples products including food & drug "
        "retailing companies."
    ),
    "Health Care": (
        "The Health Care Sector includes health care providers and services, "
        "companies that manufacture and distribute health care equipment and "
        "supplies, health care technology firms, and companies involved in "
        "pharmaceuticals and biotechnology research, development, production, "
        "and marketing."
    ),
    "Financials": (
        "The Financials Sector contains companies engaged in banking, financial "
        "services, consumer finance, capital markets, and insurance activities. "
        "It also includes financial exchanges, data services, and mortgage real "
        "estate investment trusts."
    ),
    "Information Technology": (
        "The Information Technology Sector comprises companies offering "
        "software and technology services, and manufacturers and distributors "
        "of technology hardware and equipment including communications "
        "equipment, computers, peripherals, and semiconductor products."
    ),
    "Communication Services": (
        "The Communication Services Sector includes companies that facilitate "
        "communication and offer related content and information through various "
        "mediums, including telecommunications, media and entertainment, and "
        "interactive services and platforms."
    ),
    "Utilities": (
        "The Utilities Sector comprises utility companies such as electric, "
        "gas, and water utilities, as well as independent power producers, "
        "energy traders, and companies involved in renewable generation."
    ),
    "Real Estate": (
        "The Real Estate Sector contains companies engaged in real estate "
        "development and operations as well as firms offering real estate-"
        "related services, including equity real estate investment trusts."
    ),
}

In [ ]:
import pandas as pd
import json

df_raw = pd.read_parquet("/kaggle/input/news-sentiment/dataset.parquet")

with open("/kaggle/input/news-sentiment/sector_sentiment_quotations.json", "r") as f:
    sector_sentiment_quotations = json.load(f)

gpt_testset_ids = []
for sector in sector_sentiment_quotations:
    for expected_sentiment in sector_sentiment_quotations[sector]:
        for _ in sector_sentiment_quotations[sector][expected_sentiment]:
            for quote in sector_sentiment_quotations[sector][expected_sentiment][_]:
                if quote["label"] == expected_sentiment:
                    gpt_testset_ids.append(quote["gkrecordid"] + " (" + sector + ")")

train_df = df_raw[~df_raw["GKGRECORDID"].isin(gpt_testset_ids)].copy()
min_count = train_df.value_counts(["sector", "majority_sentiment"]).values.min()

df = (
    train_df
    .groupby(["sector", "majority_sentiment"])
    .apply(lambda x: x.sample(min_count, random_state=SEED_SENTIMENT_NEWS_V2_TFIDF_EMBEDDINGS))
    .reset_index(drop=True)
)

# =========================================================
# FINBERT + TFIDF SECTOR FEATURE CLASSIFIER
# =========================================================

import os, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, AdamW
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

# ================= CONFIG =================
SEED = SEED_SENTIMENT_NEWS_V2_TFIDF_EMBEDDINGS
N_SPLITS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ProsusAI/finbert"
MAX_LEN = 128
os.makedirs("results", exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ================= LABELS =================
label2id = {l:i for i,l in enumerate(sorted(df["majority_sentiment"].unique()))}
df["label"] = df["majority_sentiment"].map(label2id)

# ================= SECTOR DESCRIPTIONS =================
sector_desc_df = pd.DataFrame({
    "sector": sector_desc.keys(),
    "sector_description": sector_desc.values()
})

# ================= DATASET =================
class SentDataset(Dataset):
    def __init__(self, df, tokenizer, tfidf_cols):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.tfidf_cols = tfidf_cols

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["quotation"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "tfidf": torch.tensor(row[self.tfidf_cols].values, dtype=torch.float),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
        }

# ================= MODEL =================
class FinBERTWithTFIDF(nn.Module):
    def __init__(self, num_labels, tfidf_dim, proj_dim, dropout, unfrozen):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)


        for name, param in self.bert.named_parameters():
            if "encoder.layer." in name:
                layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
        
                if layer_num < unfrozen:        # freeze lower layers
                    param.requires_grad = False
                else:                    # unfreeze 9, 10, 11...
                    param.requires_grad = True
            else:
                param.requires_grad = False  # embeddings, pooler etc. (optional)

        self.tfidf_proj = nn.Sequential(
            nn.LayerNorm(tfidf_dim),
            nn.Linear(tfidf_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size + proj_dim, num_labels)

    def forward(self, input_ids, attention_mask, tfidf):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0]
        tfidf_feat = self.tfidf_proj(tfidf)
        x = torch.cat([cls, tfidf_feat], dim=1)
        x = self.dropout(x)
        return self.classifier(x)

# ================= EVAL =================
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, losses = [], [], []
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    with torch.no_grad():
        for batch in loader:
            for k in batch:
                batch[k] = batch[k].to(DEVICE)
            logits = model(batch["input_ids"], batch["attention_mask"], batch["tfidf"])
            loss = loss_fn(logits, batch["label"])
            losses.append(loss.item())
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(batch["label"].cpu().tolist())

    return f1_score(all_labels, all_preds, average="macro"), np.mean(losses)

# ================= HYPERPARAM GRID =================
used_grid = [
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":5e-5, "dropout":0.3, "proj_dim":32, "tfidf_dim":100,"weight_decay": 0.0, "bs":32, "epochs":7},
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":5e-5, "dropout":0.3, "proj_dim":64, "tfidf_dim":200,"weight_decay": 0.0, "bs":32, "epochs":7},
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":5e-5, "dropout":0.2, "proj_dim":64, "tfidf_dim":300,"weight_decay":0.01, "bs":32, "epochs":7},
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":5e-5, "dropout":0.3, "proj_dim":32, "tfidf_dim":100, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":5e-5, "dropout":0.3, "proj_dim":64, "tfidf_dim":200, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":5e-5, "dropout":0.4, "proj_dim":32, "tfidf_dim":200, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":5e-5, "dropout":0.3, "proj_dim":64, "tfidf_dim":500, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":1e-5, "dropout":0.5, "proj_dim":64, "tfidf_dim":1000,"weight_decay":0.0, "bs":32, "epochs":7},
    {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":1e-5, "dropout":0.3, "proj_dim":64, "tfidf_dim":200, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":9, "bert_lr":5e-6,"head_lr":5e-5, "dropout":0.3, "proj_dim":32, "tfidf_dim":100, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":9, "bert_lr":5e-6,"head_lr":5e-5, "dropout":0.3, "proj_dim":64, "tfidf_dim":200, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":8, "bert_lr":5e-6,"head_lr":5e-5, "dropout":0.4, "proj_dim":32, "tfidf_dim":200, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":9, "bert_lr":5e-6,"head_lr":5e-5, "dropout":0.3, "proj_dim":64, "tfidf_dim":500, "weight_decay":0.0, "bs":32, "epochs":6},
    {"unfrozen_gte":8, "bert_lr":5e-6,"head_lr":1e-5, "dropout":0.5, "proj_dim":64, "tfidf_dim":1000,"weight_decay":0.0, "bs":32, "epochs":7},
    {"unfrozen_gte":9, "bert_lr":5e-6,"head_lr":1e-5, "dropout":0.3, "proj_dim":64, "tfidf_dim":200, "weight_decay":0.0, "bs":32, "epochs":6},
]

grid = [

]

# ================= TRAINING =================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for hp in grid:

    # ---- TF-IDF BUILD (depends on hp) ----
    tfidf = TfidfVectorizer(max_features=hp["tfidf_dim"], ngram_range=(1,2), stop_words="english")
    sector_tfidf = tfidf.fit_transform(sector_desc_df["sector_description"])
    sector_tfidf = pd.DataFrame(sector_tfidf.toarray(), index=sector_desc_df["sector"])
    df_hp = df.merge(sector_tfidf, left_on="sector", right_index=True)
    tfidf_cols = sector_tfidf.columns.tolist()

    combo_key = str(hp).replace(" ", "")
    combo_results = []
    learning_curve = {e: {"train": [], "val": []} for e in range(1, hp["epochs"]+1)}

    for fold, (train_idx, test_idx) in enumerate(skf.split(df_hp, df_hp["label"])):

        fold_train = df_hp.iloc[train_idx]
        fold_val = df_hp.iloc[test_idx]

        train_loader = DataLoader(SentDataset(fold_train, tokenizer, tfidf_cols), batch_size=hp["bs"], shuffle=True)
        val_loader = DataLoader(SentDataset(fold_val, tokenizer, tfidf_cols), batch_size=hp["bs"])

        model = FinBERTWithTFIDF(len(label2id), len(tfidf_cols), hp["proj_dim"], hp["dropout"], hp["unfrozen_gte"]).to(DEVICE)

        opt = AdamW([
            {"params": model.bert.encoder.layer[-2:].parameters(), "lr": hp["bert_lr"], "weight_decay": hp["weight_decay"]},
            {"params": model.tfidf_proj.parameters(), "lr": hp["head_lr"], "weight_decay": hp["weight_decay"]},
            {"params": model.classifier.parameters(), "lr": hp["head_lr"], "weight_decay": hp["weight_decay"]},
        ])

        loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

        for epoch in range(1, hp["epochs"]+1):
            model.train()
            epoch_train_losses = []

            for batch in train_loader:
                for k in batch: batch[k] = batch[k].to(DEVICE)
                opt.zero_grad()
                logits = model(batch["input_ids"], batch["attention_mask"], batch["tfidf"])
                loss = loss_fn(logits, batch["label"])
                loss.backward()
                opt.step()
                epoch_train_losses.append(loss.item())

            train_loss = np.mean(epoch_train_losses)
            val_f1, val_loss = evaluate(model, val_loader)
            learning_curve[epoch]["train"].append(train_loss)
            learning_curve[epoch]["val"].append(val_loss)

        train_f1, _ = evaluate(model, train_loader)
        combo_results.append({"fold": fold, "train_macro_f1": train_f1, "val_macro_f1": val_f1})

    print(f"Finished HP: {hp}")
    
    # Save JSON
    lc_stats = {
        e: {
            "train_mean": float(np.mean(v["train"])),
            "train_std":  float(np.std(v["train"])),
            "val_mean":   float(np.mean(v["val"])),
            "val_std":    float(np.std(v["val"]))
        }
        for e, v in learning_curve.items()
    }

    with open(f"results/{combo_key}_results.json", "w") as f:
        json.dump({"fold_results": combo_results, "learning_curve": lc_stats}, f, indent=2)

    # Plot learning curve
    epochs = sorted(lc_stats.keys())
    train_means = [lc_stats[e]["train_mean"] for e in epochs]
    train_stds  = [lc_stats[e]["train_std"] for e in epochs]
    val_means = [lc_stats[e]["val_mean"] for e in epochs]
    val_stds  = [lc_stats[e]["val_std"] for e in epochs]

    plt.figure()
    plt.plot(epochs, train_means, marker="o", label="Train Loss")
    plt.fill_between(epochs, np.array(train_means)-np.array(train_stds), np.array(train_means)+np.array(train_stds), alpha=0.2)
    plt.plot(epochs, val_means, marker="o", label="Validation Loss")
    plt.fill_between(epochs, np.array(val_means)-np.array(val_stds), np.array(val_means)+np.array(val_stds), alpha=0.2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Learning Curve\n{combo_key}")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"results/{combo_key}_learning_curve.png", bbox_inches="tight")
    plt.show()
    plt.close()

print("\n✅ Done. Results and plots saved in /results/")

print("\n✅ Done. TF-IDF hybrid training complete.")
